In [ ]:
from pathlib import Path
import copy
import json
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

BASE_DIR = Path('.')
DATA_PATH = BASE_DIR / 'data' / 'S12-hw-dataset.csv'
ARTIFACTS_DIR = BASE_DIR / 'artifacts'
FIGURES_DIR = ARTIFACTS_DIR / 'figures'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('device =', DEVICE)
print('data_path =', DATA_PATH.resolve())

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing dataset: {DATA_PATH}. Put S12-hw-dataset.csv into data/.')

df = pd.read_csv(DATA_PATH)
if 'date' not in df.columns or 'target' not in df.columns:
    raise ValueError('Dataset must contain date and target columns.')

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df['target'] = pd.to_numeric(df['target'], errors='coerce')
df = df.dropna(subset=['target']).reset_index(drop=True)

n_total = len(df)
train_end = int(n_total * 0.70)
val_end = int(n_total * 0.85)
train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print('shape =', df.shape)
print('train/val/test =', len(train_df), len(val_df), len(test_df))
display(df.head())

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df['date'], df['target'], label='target', linewidth=1.5)
ax.axvspan(df.loc[0, 'date'], df.loc[train_end - 1, 'date'], alpha=0.12, color='tab:green', label='train')
ax.axvspan(df.loc[train_end, 'date'], df.loc[val_end - 1, 'date'], alpha=0.12, color='tab:orange', label='val')
ax.axvspan(df.loc[val_end, 'date'], df.loc[n_total - 1, 'date'], alpha=0.12, color='tab:red', label='test')
ax.set_title('Temporal split')
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'series_split.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def calc_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    denom = np.clip(np.abs(y_true), 1e-8, None)
    mape = float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)
    return {'mae': float(mae), 'rmse': float(rmse), 'mape': float(mape)}


def add_features(frame):
    out = frame.copy()
    out['lag_1'] = out['target'].shift(1)
    out['lag_7'] = out['target'].shift(7)
    out['lag_14'] = out['target'].shift(14)
    out['lag_28'] = out['target'].shift(28)
    out['roll_mean_7'] = out['target'].shift(1).rolling(7).mean()
    out['roll_mean_14'] = out['target'].shift(1).rolling(14).mean()
    out['dayofweek'] = out['date'].dt.dayofweek
    out['month'] = out['date'].dt.month
    return out


feat_df = add_features(df).dropna().reset_index(drop=True)
feat_train = feat_df[feat_df['date'] <= train_df['date'].iloc[-1]].copy()
feat_val = feat_df[(feat_df['date'] >= val_df['date'].iloc[0]) & (feat_df['date'] <= val_df['date'].iloc[-1])].copy()
feat_test = feat_df[feat_df['date'] >= test_df['date'].iloc[0]].copy()
feature_cols = ['lag_1', 'lag_7', 'lag_14', 'lag_28', 'roll_mean_7', 'roll_mean_14', 'dayofweek', 'month']


def naive_last(frame):
    return frame['lag_1'].to_numpy()


def moving_average(frame):
    return frame['roll_mean_7'].to_numpy()


ridge_scaler = StandardScaler()
X_train = ridge_scaler.fit_transform(feat_train[feature_cols])
y_train = feat_train['target'].to_numpy()
X_val = ridge_scaler.transform(feat_val[feature_cols])
X_test = ridge_scaler.transform(feat_test[feature_cols])

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

baseline_rows = []
for experiment_id, title, val_pred, test_pred, notes in [
    ('B1', 'naive-last', naive_last(feat_val), naive_last(feat_test), 'predict t-1'),
    ('B2', 'moving-average', moving_average(feat_val), moving_average(feat_test), 'predict rolling mean(7)'),
    ('B3', 'ridge-lags', ridge.predict(X_val), ridge.predict(X_test), 'Ridge on lag and calendar features'),
]:
    val_metrics = calc_metrics(feat_val['target'], val_pred)
    test_metrics = calc_metrics(feat_test['target'], test_pred)
    baseline_rows.append({
        'experiment_id': experiment_id,
        'task': 'timeseries-forecasting',
        'dataset': 'S12-hw-dataset.csv',
        'seed': SEED,
        'split_summary': '70/15/15 temporal split',
        'window_size': 28,
        'horizon': 1,
        'model_summary': title,
        'features_summary': 'target lags + calendar' if experiment_id == 'B3' else 'target history only',
        'scaler': 'StandardScaler' if experiment_id == 'B3' else '',
        'optimizer': '',
        'lr': '',
        'epochs_trained': 0,
        'best_val_mae': val_metrics['mae'],
        'best_val_rmse': val_metrics['rmse'],
        'best_val_mape': val_metrics['mape'],
        'test_mae': test_metrics['mae'],
        'test_rmse': test_metrics['rmse'],
        'test_mape': test_metrics['mape'],
        'notes': notes,
    })

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(baseline_df['experiment_id'], baseline_df['best_val_mae'], color=['#5B8FF9', '#61DDAA', '#65789B'])
axes[0].set_title('Validation MAE')
axes[0].grid(axis='y', alpha=0.2)
axes[1].bar(baseline_df['experiment_id'], baseline_df['best_val_rmse'], color=['#5B8FF9', '#61DDAA', '#65789B'])
axes[1].set_title('Validation RMSE')
axes[1].grid(axis='y', alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'baselines_compare.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
WINDOW_SIZE = 28
HORIZON = 1
BATCH_SIZE = 32
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.2
LR = 5e-4
MAX_EPOCHS = 30

series = df['target'].to_numpy(dtype=np.float32)
train_mean = float(train_df['target'].mean())
train_std = float(train_df['target'].std() + 1e-8)
series_scaled = (series - train_mean) / train_std


class WindowDataset(Dataset):
    def __init__(self, series_scaled, raw_series, target_indices, window_size):
        self.series_scaled = series_scaled
        self.raw_series = raw_series
        self.target_indices = target_indices
        self.window_size = window_size

    def __len__(self):
        return len(self.target_indices)

    def __getitem__(self, idx):
        target_idx = self.target_indices[idx]
        x = self.series_scaled[target_idx - self.window_size:target_idx]
        y_scaled = self.series_scaled[target_idx]
        y_raw = self.raw_series[target_idx]
        return (
            torch.tensor(x, dtype=torch.float32).unsqueeze(-1),
            torch.tensor(y_scaled, dtype=torch.float32),
            torch.tensor(y_raw, dtype=torch.float32),
        )


all_indices = np.arange(WINDOW_SIZE, len(df))
train_indices = all_indices[all_indices < train_end]
val_indices = all_indices[(all_indices >= train_end) & (all_indices < val_end)]
test_indices = all_indices[all_indices >= val_end]

train_ds = WindowDataset(series_scaled, series, train_indices, WINDOW_SIZE)
val_ds = WindowDataset(series_scaled, series, val_indices, WINDOW_SIZE)
test_ds = WindowDataset(series_scaled, series, test_indices, WINDOW_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
xb, yb, yraw = next(iter(train_loader))
print('x.shape =', tuple(xb.shape), 'y.shape =', tuple(yb.shape))


class GRUForecast(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:, -1, :]).squeeze(-1)


model = GRUForecast(hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT).to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)


def evaluate_model(model, loader):
    model.eval()
    losses = []
    preds = []
    targets = []
    with torch.no_grad():
        for xb, yb, yraw in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            pred_scaled = model(xb)
            loss = criterion(pred_scaled, yb)
            pred_raw = pred_scaled.cpu().numpy() * train_std + train_mean
            losses.append(float(loss.item()))
            preds.extend(pred_raw.tolist())
            targets.extend(yraw.numpy().tolist())
    result = calc_metrics(targets, preds)
    result['loss'] = float(np.mean(losses)) if losses else float('nan')
    result['preds'] = np.asarray(preds)
    result['targets'] = np.asarray(targets)
    return result


history = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_rmse': []}
best_state = None
best_val_loss = float('inf')
best_epoch = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    batch_losses = []
    for xb, yb, _ in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        batch_losses.append(float(loss.item()))

    train_loss = float(np.mean(batch_losses))
    val_result = evaluate_model(model, val_loader)
    scheduler.step(val_result['loss'])

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_result['loss'])
    history['val_mae'].append(val_result['mae'])
    history['val_rmse'].append(val_result['rmse'])

    if val_result['loss'] < best_val_loss:
        best_val_loss = val_result['loss']
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())

    if epoch == 1 or epoch % 5 == 0 or epoch == MAX_EPOCHS:
        print('epoch={:02d} train_loss={:.4f} val_loss={:.4f} val_mae={:.4f}'.format(epoch, train_loss, val_result['loss'], val_result['mae']))

model.load_state_dict(best_state)
torch.save(model.state_dict(), ARTIFACTS_DIR / 'best_gru.pt')
val_result = evaluate_model(model, val_loader)
test_result = evaluate_model(model, test_loader)
print('best_epoch =', best_epoch)
print('val metrics =', {k: round(v, 4) for k, v in val_result.items() if k in ['mae', 'rmse', 'mape']})
print('test metrics =', {k: round(v, 4) for k, v in test_result.items() if k in ['mae', 'rmse', 'mape']})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='train_loss')
axes[0].plot(history['val_loss'], label='val_loss')
axes[0].set_title('GRU loss')
axes[0].legend()
axes[0].grid(alpha=0.2)
axes[1].plot(history['val_mae'], label='val_mae')
axes[1].plot(history['val_rmse'], label='val_rmse')
axes[1].set_title('GRU validation metrics')
axes[1].legend()
axes[1].grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'gru_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

test_dates = df.loc[test_indices, 'date'].to_numpy()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(test_dates, test_result['targets'], label='actual', linewidth=1.8)
ax.plot(test_dates, test_result['preds'], label='gru_pred', linewidth=1.5)
ax.set_title('Best forecast on test')
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'best_forecast_test.png', dpi=150, bbox_inches='tight')
plt.show()

gru_row = {
    'experiment_id': 'R1',
    'task': 'timeseries-forecasting',
    'dataset': 'S12-hw-dataset.csv',
    'seed': SEED,
    'split_summary': '70/15/15 temporal split',
    'window_size': WINDOW_SIZE,
    'horizon': HORIZON,
    'model_summary': 'GRU(hidden=64, layers=2, dropout=0.2)',
    'features_summary': 'univariate target windows',
    'scaler': 'train mean/std',
    'optimizer': 'Adam',
    'lr': LR,
    'epochs_trained': best_epoch,
    'best_val_mae': val_result['mae'],
    'best_val_rmse': val_result['rmse'],
    'best_val_mape': val_result['mape'],
    'test_mae': test_result['mae'],
    'test_rmse': test_result['rmse'],
    'test_mape': test_result['mape'],
    'notes': 'best checkpoint by validation loss',
}

runs_df = pd.concat([baseline_df, pd.DataFrame([gru_row])], ignore_index=True)
runs_df.to_csv(ARTIFACTS_DIR / 'runs.csv', index=False)

config = {
    'experiment_id': 'R1',
    'model': 'gru-forecast',
    'seed': SEED,
    'window_size': WINDOW_SIZE,
    'architecture': {
        'input_size': 1,
        'hidden_size': HIDDEN_SIZE,
        'num_layers': NUM_LAYERS,
        'dropout': DROPOUT,
    },
    'training': {
        'batch_size': BATCH_SIZE,
        'learning_rate': LR,
        'epochs': MAX_EPOCHS,
        'best_epoch': best_epoch,
        'optimizer': 'Adam',
        'loss': 'MSELoss',
        'scheduler': 'ReduceLROnPlateau',
    },
    'features': {
        'target_column': 'target',
        'date_column': 'date',
        'horizon': HORIZON,
    },
    'scaler': {
        'name': 'train_mean_std',
        'train_mean': train_mean,
        'train_std': train_std,
    },
}
with open(ARTIFACTS_DIR / 'best_gru_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

final_eval = {
    'best_epoch': best_epoch,
    'val': {k: float(val_result[k]) for k in ['mae', 'rmse', 'mape']},
    'test': {k: float(test_result[k]) for k in ['mae', 'rmse', 'mape']},
}
with open(ARTIFACTS_DIR / 'final_test_evaluation.json', 'w', encoding='utf-8') as f:
    json.dump(final_eval, f, ensure_ascii=False, indent=2)

display(runs_df)